# HuskySwap student: two-stage LoRA fine-tuning of Qwen3-4B

Trains a LoRA adapter on the teacher's data from `npm run datagen`:

1. **Stage 1**: `stage1.jsonl`, question → SQL (the SQL node's first turn, ending in `submit_sql`)
2. **Stage 2**: `stage2.jsonl`, every model call of the teacher's kept runs (ambiguity check, the SQL tool loop, the answer), mixed with 15% of stage 1 so it isn't forgotten

Loss is only on the model's own turns (assistant messages), not the prompts or tool results.

**Before you start:** upload `ml/datasets/stage1.jsonl` and `ml/datasets/stage2.jsonl` from the project to Google Drive, in a folder `MyDrive/huskyswap/datasets/`.

**Runtime:** A100. Adapters are saved to `MyDrive/huskyswap/adapters/` after each stage, so a disconnect doesn't lose finished work.

In [ ]:
# 1. GPU, Drive, settings
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/huskyswap"
BASE_MODEL = "unsloth/Qwen3-4B-Instruct-2507"  # same weights as Qwen/Qwen3-4B-Instruct-2507
RUN = "r1"                 # name for this training run; adapters go to adapters/stage{1,2}-RUN
MAX_LEN = 8192             # longest example kept (tokens)
LORA_RANK = 32
STAGE1 = dict(epochs=2, lr=2e-4)
STAGE2 = dict(epochs=2, lr=1e-4, stage1_mix=0.15)
SEED = 7

In [ ]:
# 2. Install Unsloth (brings TRL, PEFT, Transformers)
!pip install -q --upgrade unsloth
import unsloth, trl, transformers
print("unsloth", unsloth.__version__, "| trl", trl.__version__, "| transformers", transformers.__version__)

In [ ]:
# 3. Load the base model and add a LoRA adapter
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_LEN, dtype=torch.bfloat16, load_in_4bit=False)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_RANK, lora_alpha=2 * LORA_RANK, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED)
model.print_trainable_parameters()

In [ ]:
# 4. Load the datasets and render them with Qwen's chat template (tools included)
import json, random
from datasets import Dataset

def load_jsonl(name):
    with open(f"{ROOT}/datasets/{name}") as f:
        return [json.loads(line) for line in f if line.strip()]

def render(example):
    messages = []
    for m in example["messages"]:
        m = dict(m)
        if m.get("content") is None:
            m["content"] = ""
        messages.append(m)
    return tokenizer.apply_chat_template(messages, tools=example.get("tools"), tokenize=False)

def build(rows, label):
    texts = [render(r) for r in rows]
    lengths = [len(tokenizer(t).input_ids) for t in texts]
    kept = [t for t, n in zip(texts, lengths) if n <= MAX_LEN]
    print(f"{label}: {len(kept)} examples (dropped {len(texts) - len(kept)} longer than {MAX_LEN} tokens), median {sorted(lengths)[len(lengths) // 2]} tokens")
    return kept

random.seed(SEED)
stage1_rows, stage2_rows = load_jsonl("stage1.jsonl"), load_jsonl("stage2.jsonl")

# Hold out 5% of questions (by original question, so variants don't leak) to watch generalization
originals = sorted({r["meta"]["qid"].split("~")[0] for r in stage1_rows + stage2_rows})
held_out = set(random.sample(originals, max(1, len(originals) // 20)))
is_held = lambda r: r["meta"]["qid"].split("~")[0] in held_out

s1_train = build([r for r in stage1_rows if not is_held(r)], "stage 1 train")
s1_val = build([r for r in stage1_rows if is_held(r)], "stage 1 held-out")
s2_train = build([r for r in stage2_rows if not is_held(r)], "stage 2 train")
s2_val = build([r for r in stage2_rows if is_held(r)], "stage 2 held-out")
print("\nexample (end):\n", s1_train[0][-600:])

In [ ]:
# 5. Trainer helper: loss only on the assistant's turns
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

def train(train_texts, val_texts, epochs, lr, out):
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=Dataset.from_dict({"text": train_texts}),
        eval_dataset=Dataset.from_dict({"text": val_texts}) if val_texts else None,
        args=SFTConfig(
            dataset_text_field="text", max_seq_length=MAX_LEN, packing=False,
            per_device_train_batch_size=2, gradient_accumulation_steps=8,
            num_train_epochs=epochs, learning_rate=lr, lr_scheduler_type="cosine", warmup_steps=10,
            bf16=True, logging_steps=10, eval_strategy="epoch" if val_texts else "no",
            save_strategy="no", output_dir=f"/content/out-{out}", report_to="none", seed=SEED))
    # Qwen renders tool results as user turns, so masking user turns also masks tool output
    trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
    stats = trainer.train()
    if val_texts:
        print("held-out loss:", trainer.evaluate()["eval_loss"])
    model.save_pretrained(f"{ROOT}/adapters/{out}")
    tokenizer.save_pretrained(f"{ROOT}/adapters/{out}")
    print(f"saved {ROOT}/adapters/{out} after {stats.metrics['train_runtime'] / 60:.1f} min")

In [ ]:
# 6. Stage 1: question -> SQL
train(s1_train, s1_val, STAGE1["epochs"], STAGE1["lr"], f"stage1-{RUN}")

In [ ]:
# 7. Stage 2: the teacher's full agent traces, plus some stage 1 so it isn't forgotten.
# Questions that should get a clarifying question are rare (~2% of ambiguity checks), so they
# are repeated CLARIFY_REPEAT times; otherwise the student learns to almost never ask.
CLARIFY_REPEAT = 5
clarify_extra = build([r for r in stage2_rows if not is_held(r) and r["meta"]["expect"] == "clarify"], "clarify examples") * (CLARIFY_REPEAT - 1)
mix = random.sample(s1_train, int(len(s1_train) * STAGE2["stage1_mix"]))
stage2_texts = s2_train + clarify_extra + mix
random.shuffle(stage2_texts)
print(f"stage 2 training set: {len(stage2_texts)} examples")
train(stage2_texts, s2_val, STAGE2["epochs"], STAGE2["lr"], f"stage2-{RUN}")

In [ ]:
# 8. Quick look: the trained model's SQL for a held-out question (generation stops at its first turn)
FastLanguageModel.for_inference(model)
example = next(r for r in stage1_rows if is_held(r))
prompt = tokenizer.apply_chat_template(example["messages"][:2], tools=example["tools"], tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=400, do_sample=False)
print("question:", example["messages"][1]["content"].split("Write the SQL for the question to answer: ")[-1])
print("\nstudent:", tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=False)[:800])
print("\nteacher:", example["messages"][2]["tool_calls"][0]["function"]["arguments"][:800])
print(f"\nNext: serve it with serve_student.ipynb, ADAPTER = '{ROOT}/adapters/stage2-{RUN}', VERSION = 'stage2-{RUN}'")